# Week 10 · Day 1 — Transfer Learning: Feature Extraction

**The idea:** don't train from scratch — reuse a model already trained on millions of images.

- A CNN pretrained on **ImageNet** (1.2M images) already knows edges, textures, shapes.
- Those features work for almost any image task — including our emotions.
- Today: **freeze** the pretrained model, train only a **new head** on FER2013.
- Then compare it to our from-scratch Week 9 CNN on the same data.

> **Feature extraction** = freeze the backbone, train a new classifier head.  (Fine-tuning comes tomorrow.)

> **Kaggle GPU:** Settings → Accelerator → GPU, then add the FER2013 dataset via Add Input.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

## 1. Load FER2013 with a custom Dataset

- Same path-based `Dataset` pattern as Week 9.
- **New for pretrained models:** each image is resized to **224×224**, converted **grayscale → RGB (3 channels)**, and **normalized with ImageNet stats**.
- `torchvision.transforms` does all of this in one pipeline.

In [ ]:
# Kaggle path — set to the folder containing train/ and test/
DATA_DIR = "/kaggle/input/fer2013/train"   # e.g. .../fer2013/train and .../fer2013/test
TRAIN_DIR = DATA_DIR
TEST_DIR  = DATA_DIR.replace("train", "test")

class_names = sorted([d for d in os.listdir(TRAIN_DIR)
                      if os.path.isdir(os.path.join(TRAIN_DIR, d))])
n_classes = len(class_names)
print("classes:", class_names)

In [ ]:
# ImageNet preprocessing — required so inputs match what the pretrained model expects
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

tf = transforms.Compose([
    transforms.Resize((224, 224)),                 # model's expected input size
    transforms.Grayscale(num_output_channels=3),   # 1 channel -> 3 (repeat)
    transforms.ToTensor(),                          # -> (3, 224, 224), 0-1
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

In [ ]:
class FERDataset(Dataset):
    """Stores only paths; loads + transforms one image at a time."""
    def __init__(self, split_dir, transform):
        self.transform = transform
        self.paths, self.labels = [], []
        for i, c in enumerate(class_names):
            folder = os.path.join(split_dir, c)
            for f in os.listdir(folder):
                if f.lower().endswith((".jpg", ".jpeg", ".png")):
                    self.paths.append(os.path.join(folder, f))
                    self.labels.append(i)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("L")   # FER2013 is grayscale
        return self.transform(img), self.labels[idx]

train_ds = FERDataset(TRAIN_DIR, tf)
test_ds  = FERDataset(TEST_DIR,  tf)
print("train:", len(train_ds), " test:", len(test_ds))

In [ ]:
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True,
                          num_workers=2, pin_memory=(device.type == "cuda"))
test_loader  = DataLoader(test_ds,  batch_size=64, shuffle=False,
                          num_workers=2, pin_memory=(device.type == "cuda"))

## 2. Load a pretrained model & inspect it

- `resnet18(weights=...)` downloads a model pretrained on ImageNet.
- Structure: a big **feature-extractor body** + a final **head** (`fc`) built for 1000 ImageNet classes.

In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

print("final layer (head) built for ImageNet's 1000 classes:")
print(" ", model.fc)

## 3. Freeze the backbone, replace the head

- **Freeze** every parameter (`requires_grad = False`) → the backbone won't train.
- **Replace `fc`** with a fresh `Linear` for our 7 emotions → its params train by default.
- Result: only the tiny new head learns — fast, and works with little data.

In [ ]:
# 1. freeze the whole pretrained model
for p in model.parameters():
    p.requires_grad = False

# 2. replace the head with a new one for 7 classes (new layer -> trainable)
model.fc = nn.Linear(model.fc.in_features, n_classes)

# 3. move to GPU
model = model.to(device)

# check: only the head trains
trainable = [n for n, p in model.named_parameters() if p.requires_grad]
print("trainable params:", trainable)

## 4. Train only the head

- Same four-move loop as always: forward → loss → backward → update.
- **Only pass the head's params to the optimizer** — the backbone is frozen.
- Batches move to the GPU with `.to(device)`.

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=0.001)   # head only

@torch.no_grad()
def accuracy(model, loader):
    model.eval()
    correct = total = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        correct += (model(xb).argmax(1) == yb).sum().item()
        total += len(yb)
    return correct / total

In [ ]:
EPOCHS = 5
for epoch in range(EPOCHS):
    model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        loss = loss_fn(model(xb), yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f"epoch {epoch+1}/{EPOCHS}  test acc {accuracy(model, test_loader):.2%}")

## 5. Evaluate

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

model.eval()
preds, trues = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        preds.extend(model(xb.to(device)).argmax(1).cpu().tolist())
        trues.extend(yb.tolist())

tl_acc = np.mean(np.array(preds) == np.array(trues))
print(f"transfer-learning test accuracy: {tl_acc:.2%}")

cm = confusion_matrix(trues, preds)
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(ax=ax, cmap="Blues", colorbar=False, xticks_rotation=45)
plt.title(f"Feature extraction on FER2013 — {tl_acc:.1%}")
plt.tight_layout()
plt.show()

## 6. The rematch: transfer learning vs from-scratch

- Week 9's from-scratch CNN on FER2013: **~45%** (fill in your class's actual number).
- Today's frozen pretrained backbone + a tiny head should beat it — with far less training.
- **Why:** ImageNet features already capture the edges and shapes that faces are made of.

In [ ]:
scratch_acc = 0.45   # <- your Week 9 from-scratch result

plt.bar(["From scratch\n(Week 9)", "Transfer learning\n(today)"],
        [scratch_acc * 100, tl_acc * 100], color=["gray", "green"])
plt.ylabel("test accuracy (%)")
plt.title("Same data, same effort — reuse wins")
for i, v in enumerate([scratch_acc * 100, tl_acc * 100]):
    plt.text(i, v + 1, f"{v:.0f}%", ha="center")
plt.show()

## Your turn (solo task) ✍️

- Swap ResNet-18 for **MobileNet** and compare accuracy **and** speed.
- Starter below — same three steps: load → freeze → replace head.
- Report: which is more accurate? which trains faster?

In [ ]:
# ===== YOUR CODE (solo task) =====
# hint:
# m = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
# for p in m.parameters(): p.requires_grad = False
# m.classifier[1] = nn.Linear(m.classifier[1].in_features, n_classes)   # MobileNet's head
# m = m.to(device)
# ... then train m.classifier.parameters() with the same loop


## Summary

- **Transfer learning:** reuse a model pretrained on millions of images instead of starting from zero.
- **Feature extraction:** freeze the backbone, replace + train only a new head.
- Prep for pretrained models: resize to **224×224**, **grayscale → 3 channels**, normalize with **ImageNet stats**.
- Only the head's params go to the optimizer; everything else is frozen.
- Result: **beats the from-scratch CNN** on FER2013, with much less training.

**Tomorrow:** fine-tuning — unfreeze the backbone and adapt it too — plus a first look at object detection.